In [1]:
# S1 — The R0 exponents: are they clean fractions?
#
# x_q(R0) = 4/7 exactly for the 1->2 step (NB137, 37 ppm).
# What are the EXACT R0 exponents for all mass ratios?
# If they're clean fractions, we have the derivation.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])

P = [1]
for p in primes:
    P.append(P[-1] * p)

# The R0 exponent is: x(R0) = ln(mass_ratio) / ln(CP_R0)
# We need CP_R0 from the cascade.
# From NB155 S0: CP_R0(Q) = 189.1119

# But we can also compute it analytically (NB138):
# R0(ci; j1) = (2*pi*j1 + D) * exp(-kappa*ci) - D
# where D = epsilon*omega/(omega^2 + kappa^2)
epsilon = kappa
D = epsilon * omega / (omega**2 + kappa**2)
C1 = 2 * np.pi + D

print(f'=== R0 analytic parameters ===')
print(f'D = eps*omega/(omega^2 + kappa^2) = {D:.10f}')
print(f'C1 = 2*pi + D = {C1:.10f}')
print(f'kappa = {kappa:.10f}')

# CP pair crossings: gen2 at ci=11, gen3 at ci=191
ci_g2, ci_g3 = 11, 191

def r0_analytic(ci, j1):
    return (2*np.pi*j1 + D) * np.exp(-kappa * ci) - D

# R0 RMS at each crossing (j1=0 and j1=1 are the two sheets)
def rms_r0(ci):
    r0_j0 = r0_analytic(ci, 0)
    r0_j1 = r0_analytic(ci, 1)
    return np.sqrt(0.5 * (r0_j0**2 + r0_j1**2))

rms_g2 = rms_r0(ci_g2)
rms_g3 = rms_r0(ci_g3)
cp_R0_analytic = rms_g2 / rms_g3

print(f'\nAnalytic R0 RMS:')
print(f'  gen2 (ci=11): R0_j0={r0_analytic(11,0):.10f}, R0_j1={r0_analytic(11,1):.10f}')
print(f'  gen3 (ci=191): R0_j0={r0_analytic(191,0):.10f}, R0_j1={r0_analytic(191,1):.10f}')
print(f'  CP_R0 = {cp_R0_analytic:.10f}')

# The R0 exponent for 1->2 step:
x_q_R0 = np.log(20.0) / np.log(cp_R0_analytic)
print(f'\n=== R0 exponents ===')
print(f'x_q(R0) = ln(20)/ln(CP_R0) = {x_q_R0:.10f}')
print(f'4/7 = {4/7:.10f}')
print(f'Match: {abs(x_q_R0 - 4/7)/x_q_R0 * 1e6:.1f} ppm')

# Now compute R0 exponents for ALL mass ratios (using PDG values):
PDG_ratios = {
    'm_s/m_d':    20.0,
    'm_b/m_s':    44.75,
    'm_t/m_c':    135.8,
    'm_c/m_u':    588.0,
    'm_b/m_d':    895.0,
    'm_mu/m_e':   206.768,
    'm_tau/m_mu': 16.817,
}

print(f'\nR0 exponents for all mass ratios:')
print(f'{"Ratio":>12}  {"x(R0)":>12}  {"Fraction":>15}  {"Deviation":>10}')
print('-' * 55)

# For each ratio, compute x = ln(ratio) / ln(CP_R0)
for name, ratio in PDG_ratios.items():
    x = np.log(ratio) / np.log(cp_R0_analytic)
    
    # Find best rational approximation with small denominator
    best_frac = (0, 1)
    best_dev = 1e10
    for denom in range(1, 100):
        numer = round(x * denom)
        if numer > 0:
            dev = abs(x - numer/denom)
            if dev < best_dev:
                best_dev = dev
                best_frac = (numer, denom)
    
    n, d = best_frac
    frac_str = f'{n}/{d}'
    dev_ppm = abs(x - n/d) / x * 1e6
    
    # Also check quality: is the fraction "clean" (small numbers)?
    quality = 'CLEAN' if d <= 20 and dev_ppm < 1000 else ''
    
    print(f'{name:>12}  {x:12.8f}  {frac_str:>10} ({n/d:.8f})  {dev_ppm:8.0f} ppm  {quality}')

# ══════════════════════════════════════════════════════════════
# Let's look at the ANALYTIC structure more carefully.
# The R0 exponent is x = ln(mass_ratio) / ln(CP_R0).
# The CP_R0 is computed from the analytic R0 formula.
# Can we decompose x analytically?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Decomposing the R0 exponents ===')
print(f'CP_R0 = RMS(ci=11)/RMS(ci=191)')
print(f'  = sqrt[(R0(11,0)^2 + R0(11,1)^2) / (R0(191,0)^2 + R0(191,1)^2)]')
print(f'')

# At ci=191, the exponential is tiny: e^{-kappa*191} = {np.exp(-kappa*191):.2e}
# So R0(191, j1) ≈ (2*pi*j1 + D)*e^{-kappa*191} - D ≈ -D for both j1.
# RMS(191) ≈ |D| = {abs(D):.10f}
print(f'At ci=191 (gen3): e^(-kappa*191) = {np.exp(-kappa*191):.6e}')
print(f'  R0(191,0) = {r0_analytic(191,0):.10f}')
print(f'  R0(191,1) = {r0_analytic(191,1):.10f}')
print(f'  Both ≈ -D = {-D:.10f}')
print(f'  RMS(191) = {rms_g3:.10f} ≈ D = {D:.10f}')

# At ci=11, the exponential is significant: e^{-kappa*11} = {np.exp(-kappa*11):.6f}
alpha_11 = np.exp(-kappa * 11)
print(f'\nAt ci=11 (gen2): e^(-kappa*11) = {alpha_11:.10f}')
print(f'  R0(11,0) = D*(alpha-1) = {D*(alpha_11-1):.10f}')
print(f'  R0(11,1) = C1*alpha - D = {C1*alpha_11 - D:.10f}')
print(f'  RMS(11)  = {rms_g2:.10f}')

# The CP_R0 ratio is dominated by R0(11,1)/D:
# CP_R0 ≈ |R0(11,1)| / D = |C1*alpha - D| / D
# = |C1*alpha/D - 1|
# where C1/D = (2*pi + D)/D = 2*pi/D + 1
ratio_C1_D = C1 / D
print(f'\nC1/D = {ratio_C1_D:.10f}')
print(f'2*pi/D = {2*np.pi/D:.10f}')
print(f'alpha_11 = e^(-11*kappa) = e^(-11/sqrt(210)) = {alpha_11:.10f}')

# Simplified: CP_R0 ≈ C1*alpha/D (when D*alpha << C1*alpha)
# = (2*pi/D + 1) * alpha ≈ 2*pi*alpha/D (since D is small)
# = 2*pi * e^{-11*kappa} / (kappa*omega/(omega^2+kappa^2))
# = 2*pi * e^{-11*kappa} * (omega^2+kappa^2) / (kappa*omega)
# = (omega^2+kappa^2) / kappa * e^{-11*kappa}
# since 2*pi/omega = 1

simplification = (omega**2 + kappa**2) / kappa * alpha_11
print(f'\nSimplified CP_R0 ≈ (omega^2+kappa^2)/kappa * e^(-11*kappa) = {simplification:.4f}')
print(f'Actual CP_R0 = {cp_R0_analytic:.4f}')
print(f'Ratio: {simplification/cp_R0_analytic:.6f}')

# The EXACT CP_R0 involves both j1=0 and j1=1 branches.
# Let me compute it precisely.
r0_11_0 = D * (alpha_11 - 1)
r0_11_1 = C1 * alpha_11 - D
r0_191_0 = D * (np.exp(-kappa*191) - 1)
r0_191_1 = C1 * np.exp(-kappa*191) - D

cp_R0_exact = np.sqrt((r0_11_0**2 + r0_11_1**2) / (r0_191_0**2 + r0_191_1**2))
print(f'\nExact CP_R0 = sqrt[({r0_11_0:.6f}^2 + {r0_11_1:.6f}^2) / ({r0_191_0:.6f}^2 + {r0_191_1:.6f}^2)]')
print(f'            = sqrt[{r0_11_0**2 + r0_11_1**2:.6f} / {r0_191_0**2 + r0_191_1**2:.6f}]')
print(f'            = {cp_R0_exact:.10f}')

# Now: what determines x(R0) = 4/7?
# x(R0) = ln(20) / ln(CP_R0)
# = ln(20) / ln(sqrt[(r0_11_0^2 + r0_11_1^2) / (r0_191_0^2 + r0_191_1^2)])
# = 2*ln(20) / ln[(r0_11_0^2 + r0_11_1^2) / (r0_191_0^2 + r0_191_1^2)]

# The denominator (ci=191) ≈ 2*D^2 (both branches ≈ -D)
# The numerator (ci=11) = D^2*(a-1)^2 + (C1*a - D)^2 where a = e^{-11*kappa}

denom_191 = r0_191_0**2 + r0_191_1**2
numer_11 = r0_11_0**2 + r0_11_1**2

print(f'\nDenominator (2*D^2): {2*D**2:.10f}  actual: {denom_191:.10f}  ratio: {denom_191/(2*D**2):.10f}')
print(f'Numerator (exact):   {numer_11:.10f}')
print(f'Ratio N/D = {numer_11/denom_191:.10f}')
print(f'CP_R0^2 = {cp_R0_exact**2:.10f}')

# ══════════════════════════════════════════════════════════════
# The R0 exponent depends on the CROSSING POSITIONS ci=11 and ci=191.
# These positions come from the CRT structure.
# What if we compute x(R0) for DIFFERENT generation pairs?
# ══════════════════════════════════════════════════════════════

print(f'\n=== R0 analytic: cross-gen ratios ===')
# All three gen crossings in the quark Z2=0 sector:
ci_vals = {'gen1': 71, 'gen2': 11, 'gen3': 191}

for name_a, ci_a in ci_vals.items():
    for name_b, ci_b in ci_vals.items():
        if ci_a <= ci_b:
            continue
        rms_a = rms_r0(ci_a)
        rms_b = rms_r0(ci_b)
        if rms_b > 1e-10:
            ratio = rms_a / rms_b
            print(f'  {name_a}/{name_b}: ci={ci_a}/{ci_b}  ratio={ratio:.6f}  ln(ratio)={np.log(ratio):.6f}')

# The ANALYTIC R0 formula gives exact values at any crossing.
# The mass ratios might come from specific COMBINATIONS of these R0 values.

# Key: at R0, the spatial profile is analytic (NB138).
# Only j1=0 and j1=1 matter (p1=2 sheets at R0).
# The RMS at crossing ci is:
#   RMS(ci) = sqrt(0.5 * [D^2*(a-1)^2 + (C1*a - D)^2])
# where a = e^{-kappa*ci}.

# This is a known function of ci. The mass ratios at R0 are:
#   mass_ratio = [RMS(ci_a)/RMS(ci_b)]^{x(R0)}
# where x(R0) = 4/7 for the CP pair.

# What if ALL mass ratios use x(R0) = 4/7 but with DIFFERENT crossing pairs?
print(f'\n=== Can different crossing pairs give different mass ratios with x(R0)=4/7? ===')
x_R0 = 4.0 / 7.0

# CP pair (gen2/gen3): [RMS(11)/RMS(191)]^{4/7} = 20.0 (m_s/m_d)
val_23 = (rms_r0(11) / rms_r0(191)) ** x_R0
print(f'  (gen2/gen3)^(4/7) = {val_23:.4f}  [m_s/m_d = 20.0]')

# What about gen1/gen3 with 4/7?
val_13 = (rms_r0(71) / rms_r0(191)) ** x_R0
print(f'  (gen1/gen3)^(4/7) = {val_13:.4f}')

# gen2/gen1 with 4/7?
val_21 = (rms_r0(11) / rms_r0(71)) ** x_R0
print(f'  (gen2/gen1)^(4/7) = {val_21:.4f}')

# Products?
print(f'  (gen2/gen3)^(4/7) × (gen2/gen1)^(4/7) = {val_23 * val_21:.4f}  [m_b/m_d = 895?]')
print(f'  (gen2/gen3)^(4/7) × (gen1/gen3)^(4/7) = {val_23 * val_13:.4f}')
print(f'  [(gen2/gen3) × (gen2/gen1)]^(4/7) = {((rms_r0(11)/rms_r0(191)) * (rms_r0(11)/rms_r0(71)))**x_R0:.4f}')
print(f'  [gen2^2/(gen1×gen3)]^(4/7) = {((rms_r0(11)**2)/(rms_r0(71)*rms_r0(191)))**x_R0:.4f}')

# Try: mass_ratio = (gen_heavy/gen_light)^{4/7} × correction_from_other_crossings
# m_b/m_s = (gen2/gen3)^{4/7} × (gen1/gen3)^{something}?
# 44.75 = 20.0 × (gen1/gen3)^{y}?
# y = ln(44.75/20.0) / ln(val_13) ... but val_13 is from ^{4/7}

# Let me think about this differently.
# At R0, the ANALYTIC formula gives RMS(ci) as a known function.
# RMS(ci) depends on D, C1, kappa, and ci.
# D and C1 and kappa are determined by primes.
# ci is determined by CRT structure.
# So RMS(ci) is a computable function of the primes and CRT positions.

# What if mass(gen_k) ∝ 1/RMS_R0(ci_k)?
print(f'\n=== Mass ∝ 1/RMS_R0 test ===')
rms_r0_g1 = rms_r0(71)
rms_r0_g2 = rms_r0(11)
rms_r0_g3 = rms_r0(191)

# If mass ∝ 1/RMS_R0:
# m_b/m_d = RMS_R0(71)/RMS_R0(191) = (gen1/gen3 at R0)
# m_s/m_d = RMS_R0(11)/... hmm, this doesn't map cleanly.

# Actually: heavier fermion = gen3 = ci=191 = SMALLEST RMS.
# If mass ∝ 1/RMS, then m_b ∝ 1/RMS(191), m_s ∝ 1/RMS(11), m_d ∝ 1/RMS(71).
# m_b/m_s = RMS(11)/RMS(191) = CP_R0 = 189.1. But m_b/m_s = 44.75. Doesn't work.
# m_s/m_d = RMS(71)/RMS(11) = {rms_r0_g1/rms_r0_g2}. 

inv_rms = {1: 1/rms_r0_g1, 2: 1/rms_r0_g2, 3: 1/rms_r0_g3}
print(f'1/RMS_R0: gen1={inv_rms[1]:.2f}, gen2={inv_rms[2]:.2f}, gen3={inv_rms[3]:.2f}')
print(f'gen3/gen2 = {inv_rms[3]/inv_rms[2]:.2f} (m_b/m_s = 44.75)')
print(f'gen2/gen1 = {inv_rms[2]/inv_rms[1]:.2f} (m_s/m_d = 20.0)')

# Doesn't work linearly. But what about 1/RMS^{4/7}?
print(f'\n(1/RMS_R0)^(4/7):')
for g in [1, 2, 3]:
    print(f'  gen{g}: {inv_rms[g]**(4/7):.4f}')
print(f'gen3/gen2 = {(inv_rms[3]/inv_rms[2])**(4/7):.4f}')
print(f'gen2/gen1 = {(inv_rms[2]/inv_rms[1])**(4/7):.4f}')

# What about RMS^{-4/7}?
print(f'\nRMS_R0^(-4/7):')
for label, rms_val in [('gen1(ci=71)', rms_r0_g1), ('gen2(ci=11)', rms_r0_g2), ('gen3(ci=191)', rms_r0_g3)]:
    print(f'  {label}: {rms_val**(-4/7):.6f}')

m_proxy = {k: rms_r0(ci_vals[f'gen{k}'])**(-4/7) for k in [1,2,3]}
print(f'\nMass proxy ratios (RMS^(-4/7)):')
print(f'  gen3/gen2 = {m_proxy[3]/m_proxy[2]:.4f}  (m_b/m_s = 44.75)')
print(f'  gen2/gen1 = {m_proxy[2]/m_proxy[1]:.4f}  (m_s/m_d = 20.0)')
print(f'  gen3/gen1 = {m_proxy[3]/m_proxy[1]:.4f}  (m_b/m_d = 895)')

# KEY: (RMS_R0)^{-4/7} as mass proxy at each crossing gives
# the SAME exponent 4/7 applied to INDIVIDUAL crossings, not ratios.
# If this works, ALL mass ratios emerge from the spatial profile at R0.

=== R0 analytic parameters ===
D = eps*omega/(omega^2 + kappa^2) = 0.0109814099
C1 = 2*pi + D = 6.2941667171
kappa = 0.0690065559

Analytic R0 RMS:
  gen2 (ci=11): R0_j0=-0.0058410057, R0_j1=2.9353216113
  gen3 (ci=191): R0_j0=-0.0109813892, R0_j1=-0.0109695296
  CP_R0 = 189.1118676498

=== R0 exponents ===
x_q(R0) = ln(20)/ln(CP_R0) = 0.5714495813
4/7 = 0.5714285714
Match: 36.8 ppm

R0 exponents for all mass ratios:
       Ratio         x(R0)         Fraction   Deviation
-------------------------------------------------------
     m_s/m_d    0.57144958         4/7 (0.57142857)        37 ppm  CLEAN
     m_b/m_s    0.72507551       29/40 (0.72500000)       104 ppm  
     m_t/m_c    0.93683058       89/95 (0.93684211)        12 ppm  
     m_c/m_u    1.21638972      118/97 (1.21649485)        86 ppm  
     m_b/m_d    1.29652509      118/91 (1.29670330)       137 ppm  
    m_mu/m_e    1.01702650       60/59 (1.01694915)        76 ppm  
  m_tau/m_mu    0.53838381        7/13 (0.53846154)   

# NB157 — Multi-Level Spatial Structure

**Context**: NB156 showed the R3 spatial geometry (7-tooth comb + wrapping) explains
the CP pair ratio but cannot derive the inter-gen scaling factors r_bs=1.269,
r_tc=1.639. The missing information must live in the MULTI-LEVEL structure.

**Goal**: Analyze all 4 covering levels simultaneously at the generation crossings.
The covering cascade couples levels: R_k = p_{k+1}*theta_{k+1} - theta_k.
Each level has its OWN comb with its OWN spacing and wrapping pattern.
The mass might emerge from the PRODUCT of information across all 4 levels,
not from any single level.

**Key question**: At each generation crossing, we have a 4-dimensional spatial
amplitude (R0, R1, R2, R3). Does the full 4D structure at the three generation
crossings determine all mass ratios without needing separate scaling factors?

In [2]:
# S0 — Full 4-level spatial analysis at generation crossings
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
omega = 2 * np.pi
lambda_P4 = reduce(_lcm, [p - 1 for p in primes])

P = [1]
for p in primes:
    P.append(P[-1] * p)

sys0 = SolenoidSystem(primes=primes)
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()
t_eval = cis.astype(float)
res = sys0.integrate_all_branches(all_branches, t_eval, float(P4 + 1), backend='jax')

# Full branch-resolved data at each crossing and level
n_cross = len(cis)
n_branches = len(all_branches)
R_full = np.zeros((n_cross, n_branches, 4))
for idx in range(n_cross):
    for br_idx, br in enumerate(all_branches):
        for k in range(4):
            R_full[idx, br_idx, k] = res[br][idx, k]

# Wrapped RMS at each crossing and level
rms = np.zeros((n_cross, 4))
for idx in range(n_cross):
    for k in range(4):
        Rk_w = np.mod(R_full[idx, :, k], 2*np.pi)
        Rk_w[Rk_w > np.pi] -= 2*np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}
def get_idx(a3v, a7v):
    return np.where((a3 == a3v) & (a5 == 0) & (a7 == a7v))[0][0]

# Generation crossing indices (Z2=0 quarks)
idx_g1 = get_idx(1, 0)  # ci=71
idx_g2 = get_idx(1, 4)  # ci=11
idx_g3 = get_idx(1, 2)  # ci=191

print(f'Cascade integrated. {n_branches} branches, {n_cross} crossings, 4 levels.')

# ══════════════════════════════════════════════════════════════
# 1. The 4D spatial amplitude at each generation crossing
# ══════════════════════════════════════════════════════════════

print(f'\n=== 4D spatial amplitude at generation crossings ===')
print(f'{"":>8} {"R0":>10} {"R1":>10} {"R2":>10} {"R3":>10}  {"Product":>10}  {"GeoMean":>10}')
for label, idx in [('gen1', idx_g1), ('gen2', idx_g2), ('gen3', idx_g3)]:
    r = rms[idx]
    prod_r = np.prod(r)
    gmean = np.prod(r)**(1/4)
    print(f'{label:>8} {r[0]:10.6f} {r[1]:10.6f} {r[2]:10.6f} {r[3]:10.6f}  {prod_r:10.4f}  {gmean:10.6f}')

# ══════════════════════════════════════════════════════════════
# 2. Cross-generation ratios at EACH level
# ══════════════════════════════════════════════════════════════

print(f'\n=== Cross-generation ratios at each level ===')
r_g1 = rms[idx_g1]
r_g2 = rms[idx_g2]
r_g3 = rms[idx_g3]

print(f'{"Ratio":>12} {"R0":>10} {"R1":>10} {"R2":>10} {"R3":>10}')
cp_23 = r_g2 / r_g3
cp_13 = r_g1 / r_g3
cp_21 = r_g2 / r_g1
print(f'{"gen2/gen3":>12} {cp_23[0]:10.4f} {cp_23[1]:10.4f} {cp_23[2]:10.4f} {cp_23[3]:10.4f}  (CP pair)')
print(f'{"gen1/gen3":>12} {cp_13[0]:10.4f} {cp_13[1]:10.4f} {cp_13[2]:10.4f} {cp_13[3]:10.4f}')
print(f'{"gen2/gen1":>12} {cp_21[0]:10.4f} {cp_21[1]:10.4f} {cp_21[2]:10.4f} {cp_21[3]:10.4f}')

# ══════════════════════════════════════════════════════════════
# 3. The factored architecture: CP_Rk^{x_q(k)} = constant for the CP pair.
#    Does this hold for OTHER cross-gen ratios?
# ══════════════════════════════════════════════════════════════

x_q = 1.5866463961
ln_cp23 = np.log(cp_23)

print(f'\n=== Factored architecture test ===')
print(f'CP pair: CP_Rk^{{x_q(k)}} should be constant:')
for k in range(4):
    x_k = x_q * ln_cp23[3] / ln_cp23[k]
    val = cp_23[k] ** x_k
    print(f'  R{k}: CP={cp_23[k]:.4f}, x_q(R{k})={x_k:.4f}, CP^x={val:.4f}')

print(f'\ngen1/gen3: does the same factored architecture hold?')
ln_cp13 = np.log(np.maximum(cp_13, 1e-10))
for k in range(4):
    if cp_13[k] > 0.001:
        x_k_13 = x_q * ln_cp23[3] / ln_cp23[k]  # SAME factored exponent from CP pair
        val_13 = cp_13[k] ** x_k_13
        print(f'  R{k}: ratio={cp_13[k]:.4f}, (gen1/gen3)^{{x_CP(k)}}={val_13:.4f}')

print(f'\ngen2/gen1: does factored architecture hold?')
for k in range(4):
    if cp_21[k] > 0.001:
        x_k_21 = x_q * ln_cp23[3] / ln_cp23[k]
        val_21 = cp_21[k] ** x_k_21
        print(f'  R{k}: ratio={cp_21[k]:.4f}, (gen2/gen1)^{{x_CP(k)}}={val_21:.4f}')

# ══════════════════════════════════════════════════════════════
# 4. KEY QUESTION: Is there a DIFFERENT exponent for each cross-gen ratio
#    that gives a CONSTANT value across levels?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Finding level-independent exponents for each cross-gen ratio ===')
# For each ratio, find x such that ratio(R3)^x = ratio(Rk)^{x * cl_inv(k)}
# The factored architecture guarantees this for the CP pair.
# But does each cross-gen ratio have its OWN factored architecture?

# If ratio_Rk = ratio_R3^{cl(k)} for each ratio, then ALL ratios share
# the same cross-level structure. The scaling factor would then be
# r = ln(ratio_R3) / ln(CP_R3) and would be level-independent.
# This is EXACTLY what we found in S0: r_bs and r_tc have spread=0.

# So: does gen1/gen3 have the SAME cross-level structure as gen2/gen3?
print(f'Cross-level structure comparison:')
print(f'{"":>12} {"cl(0)":>10} {"cl(1)":>10} {"cl(2)":>10} {"cl(3)":>10}')

cl_23 = ln_cp23 / ln_cp23[3]  # cross-levels from CP pair
print(f'{"gen2/gen3":>12} {cl_23[0]:10.6f} {cl_23[1]:10.6f} {cl_23[2]:10.6f} {cl_23[3]:10.6f}')

if all(cp_13[k] > 0.001 for k in range(4)):
    cl_13 = np.log(cp_13) / np.log(cp_13[3])
    print(f'{"gen1/gen3":>12} {cl_13[0]:10.6f} {cl_13[1]:10.6f} {cl_13[2]:10.6f} {cl_13[3]:10.6f}')
    print(f'{"difference":>12} {(cl_23-cl_13)[0]:+10.6f} {(cl_23-cl_13)[1]:+10.6f} {(cl_23-cl_13)[2]:+10.6f} {(cl_23-cl_13)[3]:+10.6f}')

if all(cp_21[k] > 0.001 for k in range(4)):
    cl_21 = np.log(cp_21) / np.log(cp_21[3])
    print(f'{"gen2/gen1":>12} {cl_21[0]:10.6f} {cl_21[1]:10.6f} {cl_21[2]:10.6f} {cl_21[3]:10.6f}')
    print(f'{"difference":>12} {(cl_23-cl_21)[0]:+10.6f} {(cl_23-cl_21)[1]:+10.6f} {(cl_23-cl_21)[2]:+10.6f} {(cl_23-cl_21)[3]:+10.6f}')

# ══════════════════════════════════════════════════════════════
# 5. The wrapping structure at EACH level
# ══════════════════════════════════════════════════════════════

print(f'\n=== Wrapping structure at ALL levels (gen crossings) ===')
j_vals = [np.array([br[k] for br in all_branches]) for k in range(4)]

for label, idx_gen, ci_val in [('gen1', idx_g1, 71), ('gen2', idx_g2, 11), ('gen3', idx_g3, 191)]:
    print(f'\n  {label} (ci={ci_val}):')
    for k in range(4):
        Rk = R_full[idx_gen, :, k]
        n_wrap = np.sum(np.abs(Rk) > np.pi)
        max_j = primes[k] if k < len(primes) else 1  # p_{k+1} sheets at level k
        # Compute per-j_k structure
        jk = j_vals[k]
        n_sheets = int(jk.max()) + 1
        sheet_means = [np.mean(Rk[jk == j]) for j in range(n_sheets)]
        spacing = np.mean(np.diff(sheet_means)) if n_sheets > 1 else 0
        pred_spacing = 2 * np.pi * np.exp(-kappa * ci_val)
        print(f'    R{k}: {n_sheets} sheets, spacing={spacing:.4f} (pred={pred_spacing:.4f}), '
              f'wrapped={n_wrap}/{n_branches} ({n_wrap/n_branches*100:.1f}%)')

# ══════════════════════════════════════════════════════════════
# 6. PRODUCT of levels: does the 4D amplitude determine mass?
# ══════════════════════════════════════════════════════════════

print(f'\n=== Multi-level mass candidates ===')
# Try: mass ∝ product of RMS across levels
prod_g1 = np.prod(r_g1)
prod_g2 = np.prod(r_g2)
prod_g3 = np.prod(r_g3)
print(f'Product of RMS across 4 levels:')
print(f'  gen1: {prod_g1:.6f}')
print(f'  gen2: {prod_g2:.6f}')
print(f'  gen3: {prod_g3:.6f}')
print(f'  gen2/gen3: {prod_g2/prod_g3:.4f}')
print(f'  gen1/gen3: {prod_g1/prod_g3:.4f}')

# Geometric mean
gmean_g1 = np.prod(r_g1)**0.25
gmean_g2 = np.prod(r_g2)**0.25
gmean_g3 = np.prod(r_g3)**0.25
print(f'\nGeometric mean of RMS across levels:')
print(f'  gen1: {gmean_g1:.6f}')
print(f'  gen2: {gmean_g2:.6f}')
print(f'  gen3: {gmean_g3:.6f}')
print(f'  gen2/gen3: {gmean_g2/gmean_g3:.4f}')
print(f'  (gen2/gen3)^x_q: {(gmean_g2/gmean_g3)**x_q:.4f}')

# Weighted combinations
print(f'\n--- Exploring multi-level combinations ---')
# The CP ratio at each level encodes different aspects of the covering:
# R0 (p=2): innermost, strongest coupling
# R1 (p=3): chirality level
# R2 (p=5): isospin level 
# R3 (p=7): outermost, generation level
#
# Maybe different mass ratios use different LEVEL COMBINATIONS:
# m_s/m_d (1->2 down): R3 only (single generation step)
# m_b/m_s (2->3 down): R3 × R2 (generation + isospin)
# m_t/m_c (2->3 up): R2 × R1 (isospin + chirality)?

# Test: CP_R3 * CP_R2 product
cp23_r3r2 = cp_23[3] * cp_23[2]  # product of CP at R3 and R2
cp23_r3r1 = cp_23[3] * cp_23[1]
cp23_r2r1 = cp_23[2] * cp_23[1]
cp23_r3r2r1 = cp_23[3] * cp_23[2] * cp_23[1]

print(f'\nMulti-level CP products (gen2/gen3):')
print(f'  R3 alone: {cp_23[3]:.4f}  ^x_q = {cp_23[3]**x_q:.2f}  (m_s/m_d = 20)')
print(f'  R3*R2:    {cp23_r3r2:.4f}  ^x_q = {cp23_r3r2**x_q:.2f}')
print(f'  R3*R1:    {cp23_r3r1:.4f}  ^x_q = {cp23_r3r1**x_q:.2f}')
print(f'  R2*R1:    {cp23_r2r1:.4f}  ^x_q = {cp23_r2r1**x_q:.2f}')
print(f'  R3*R2*R1: {cp23_r3r2r1:.4f}  ^x_q = {cp23_r3r2r1**x_q:.2f}')
print(f'  R3*R0:    {cp_23[3]*cp_23[0]:.4f}  ^x_q = {(cp_23[3]*cp_23[0])**x_q:.2f}')

print(f'\n  PDG targets: m_b/m_s = 44.75, m_t/m_c = 135.8, m_c/m_u = 588')

# Check: sqrt(R3 * R2)
print(f'\n  sqrt(R3*R2): {np.sqrt(cp23_r3r2):.4f}  ^x_q = {np.sqrt(cp23_r3r2)**x_q:.2f}')

# What about RMS at different levels for different mass ratios?
# m_b/m_s: the 2->3 down-type step. This involves gen3 and gen2.
# The CP pair IS gen2/gen3. So CP_R3^{x_q} = 20.0 = m_s/m_d.
# And CP_R3^{2*x_q} ≈ 400 (too big for m_b/m_s = 44.75)
# But what if m_b/m_s involves a DIFFERENT level combination?

# Let's systematically search for level combinations that give the right ratios
print(f'\n=== Systematic search: which level combinations give PDG ratios? ===')
targets = {'m_b/m_s': 44.75, 'm_t/m_c': 135.8, 'm_c/m_u': 588.0}

# For each target, try CP_Ri^a * CP_Rj^b * ... with a,b in {-1,0,1}
for target_name, target_val in targets.items():
    print(f'\n  {target_name} = {target_val}:')
    best_dev = 1e10
    best_formula = ''
    for a0 in range(-1, 2):
        for a1 in range(-1, 2):
            for a2 in range(-1, 2):
                for a3 in range(-1, 2):
                    if a0 == a1 == a2 == a3 == 0:
                        continue
                    base = cp_23[0]**a0 * cp_23[1]**a1 * cp_23[2]**a2 * cp_23[3]**a3
                    if base > 0.001:
                        # Try base^x_q
                        val = base ** x_q
                        dev = abs(val - target_val) / target_val
                        if dev < 0.05:  # within 5%
                            powers = f'R0^{a0}*R1^{a1}*R2^{a2}*R3^{a3}'
                            print(f'    ({powers})^x_q = {base:.4f}^{x_q:.4f} = {val:.2f} '
                                  f'({dev*100:.2f}%)')
                            if dev < best_dev:
                                best_dev = dev
                                best_formula = powers
    if best_dev < 0.05:
        print(f'    BEST: ({best_formula})^x_q, dev={best_dev*100:.2f}%')

# ══════════════════════════════════════════════════════════════
# 7. The COMPLETE spatial mass formula
# ══════════════════════════════════════════════════════════════

# Summary of what we found
print(f'\n{"="*70}')
print(f'SUMMARY')
print(f'{"="*70}')
print(f'The spatial structure at the three generation crossings provides:')
print(f'  gen1 (ci=71):  RMS = [{r_g1[0]:.4f}, {r_g1[1]:.4f}, {r_g1[2]:.4f}, {r_g1[3]:.4f}]')
print(f'  gen2 (ci=11):  RMS = [{r_g2[0]:.4f}, {r_g2[1]:.4f}, {r_g2[2]:.4f}, {r_g2[3]:.4f}]')
print(f'  gen3 (ci=191): RMS = [{r_g3[0]:.4f}, {r_g3[1]:.4f}, {r_g3[2]:.4f}, {r_g3[3]:.4f}]')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.49s
Cascade integrated. 210 branches, 48 crossings, 4 levels.

=== 4D spatial amplitude at generation crossings ===
                 R0         R1         R2         R3     Product     GeoMean
    gen1   0.026539   0.148617   0.198686   0.585814      0.0005    0.146375
    gen2   2.075590   1.618601   1.737103   1.846494     10.7759    1.811814
    gen3   0.010975   0.027498   0.043644   0.279486      0.0000    0.043803

=== Cross-generation ratios at each level ===
       Ratio         R0         R1         R2         R3
   gen2/gen3   189.1119    58.8635    39.8014     6.6067  (CP pair)
   gen1/gen3     2.4180     5.4047     4.5524     2.0960
   gen2/gen1    78.2102    10.8911     8.7430     3.1520

=== Factored architecture test ===
CP pair: CP_Rk^{x_q(k)} should be constant:
  R0: CP=189.1119, x_q(R0)=0.5714, CP^x=20.0000
  R1: CP=58.8635, x_q(R1)=0.7351, CP^x=20.0000
  R2: CP=39.8014, x_q(R2)=0.8132, CP^x=20.0000
  